This notebook prepares merges covariate & breakpoint data, and prepares input files for fishHook

In [1]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm

In [3]:
def merge_covariates(binned_df, rt_file, gc_file, map_file, chrom_state_file, gene_density_file, breakpoint_counts, fragile_sites_file, repeats_file):
    """
    Merge binned breakpoint data with covariates
    """
    binned_df = pd.read_csv(binned_df, sep='\t')
    
    #covariate files
    rt_data = pd.read_csv(rt_file, sep='\t')
    gc_data = pd.read_csv(gc_file, sep='\t')
    map_data = pd.read_csv(map_file, sep='\t')
    chrom_state_data = pd.read_csv(chrom_state_file, sep='\t')
    gene_density_data = pd.read_csv(gene_density_file)
    fragile_sites_data = pd.read_csv(fragile_sites_file)
    repeats_data = pd.read_csv(repeats_file)

    # breakpoint data
    breakpoint_data = pd.read_csv(breakpoint_counts)
    # rename second column to 'breakpoint_count'
    breakpoint_data.rename(columns={breakpoint_data.columns[1]: 'breakpoint_count'}, inplace=True)
   
    merged_df = binned_df.copy()
    
   # merging
    merged_df = pd.merge(merged_df, rt_data[['chrom', 'start', 'end', 'replication_timing']], on=['chrom', 'start', 'end'], how='left')
    merged_df = pd.merge(merged_df, gc_data[['chrom', 'start', 'end', 'gc_content']], on=['chrom', 'start', 'end'], how='left')
    merged_df = pd.merge(merged_df, map_data[['chrom', 'start', 'end', 'mappability']], on=['chrom', 'start', 'end'], how='left')
    merged_df = pd.merge(merged_df, chrom_state_data[['chrom', 'start', 'end', 'frac_heterochromatin']], on=['chrom', 'start', 'end'], how='left')
    merged_df = pd.merge(merged_df, gene_density_data[['chrom', 'start', 'end', 'gene_density']], on=['chrom', 'start', 'end'], how='left')
    merged_df = pd.merge(merged_df, fragile_sites_data[['chrom', 'start', 'end', 'frac_fragile_site']], on=['chrom', 'start', 'end'], how='left')
    merged_df = pd.merge(merged_df, breakpoint_data[['bin_id', 'breakpoint_count']], on=['bin_id'], how='left')
    merged_df['breakpoint_count'] = merged_df['breakpoint_count'].fillna(0)
    merged_df = pd.merge(merged_df, repeats_data[['chrom', 'start', 'end', 'frac_sine', 'frac_line', 'frac_ltr']], on=['chrom', 'start', 'end'], how='left')                    

    # drop bins with missing covariate data
    print(f"Number of bins before dropping NA: {len(merged_df)}")
    merged_df = merged_df.dropna(subset=['replication_timing', 'gc_content', 'mappability', 'frac_heterochromatin', 'gene_density', 'frac_sine', 'frac_fragile_site', 'frac_line', 'frac_ltr'])
    print(f"Number of bins after dropping NA: {len(merged_df)}")
    
    # scale covariates to z-scores
    scaler = StandardScaler()
    features_to_scale = ['replication_timing','gc_content', 'mappability', 'frac_heterochromatin', 'gene_density', 'frac_sine', 'frac_fragile_site', 'frac_line', 'frac_ltr']
    merged_df[features_to_scale] = scaler.fit_transform(merged_df[features_to_scale])
    
    merged_df.to_csv('../data/merged_covariates.csv', index=False)
    
    return merged_df

merged_data = merge_covariates(
    binned_df='../data/genome_bins.bed',
    rt_file='../data/imr90_rt_binned.bed',
    gc_file='../data/gc_content.bed',
    map_file='../data/mappability.bed',
    chrom_state_file='../data/chrom_states.bed',
    gene_density_file='../data/gene_density.csv',
    breakpoint_counts='../data/breakpoints_per_bin_distribution.csv',
    fragile_sites_file='../data/fragile_sites.csv',
    repeats_file='../data/repeat_features.csv'
)

merged_data.head()

Number of bins before dropping NA: 56596
Number of bins after dropping NA: 55917


,bin_id,chrom,start,end,replication_timing,gc_content,mappability,frac_heterochromatin,gene_density,frac_fragile_site,breakpoint_count,frac_sine,frac_line,frac_ltr
0,0,1,25000,75000,0.228249,-0.310924,-7.091714,-0.462759,3.904444,1.323585,1.0,-0.727919,-0.069211,-0.209417
1,1,1,75000,125000,0.190545,-0.434277,-7.278499,-0.282773,0.505917,1.323585,0.0,-0.263260,1.076090,-0.005731
3,3,1,375000,425000,0.067359,0.583692,-10.619320,-0.615054,-1.193347,1.323585,0.0,0.022927,-0.137056,0.172940
4,4,1,425000,475000,0.049794,-0.561390,-9.024493,-0.615054,-0.060505,1.323585,0.0,-1.125178,2.496487,0.214722
6,6,1,675000,725000,0.362343,-0.454804,-8.847729,-0.615054,1.638759,1.323585,0.0,-0.699153,0.925686,0.153698


In [4]:
def prepare_data_for_fishhook(merged_df):
    """
    Prepare data files for FishHook R package
    """
    # 1. create breakpoints BED file w/ actual breakpoint locations
    breakpoints_df = pd.read_csv('../data/binned_breakpoints.csv')
    breakpoints_bed = breakpoints_df[['chrom', 'position']].copy()
    breakpoints_bed['start'] = breakpoints_bed['position'] - 1
    breakpoints_bed = breakpoints_bed.rename(columns={'position': 'end'})
    # add 'chr' prefix
    if not breakpoints_bed['chrom'].astype(str).str.startswith('chr').all():
        breakpoints_bed['chrom'] = 'chr' + breakpoints_bed['chrom'].astype(str)
    breakpoints_bed = breakpoints_bed[['chrom', 'start', 'end']]
    breakpoints_bed.to_csv('../data/breakpoints_for_fishhook.bed', sep='\t', index=False, header=False)

    # create covariates file
    covariates_df = merged_df[['chrom', 'start', 'end', 'replication_timing', 'gc_content', 'mappability', 'frac_sine', 'frac_heterochromatin', 'frac_fragile_site']]
    if not covariates_df['chrom'].astype(str).str.startswith('chr').all():
        covariates_df['chrom'] = 'chr' + covariates_df['chrom'].astype(str)
    covariates_df.to_csv('../data/covariates_for_fishhook.txt', sep='\t', index=False)

prepare_data_for_fishhook(merged_data)

/var/folders/l_/yt1nk3rj6rvgt0nnj_qtfs7m0000gn/T/ipykernel_63173/2372488843.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  covariates_df['chrom'] = 'chr' + covariates_df['chrom'].astype(str)
